In [26]:
import boto3

# Initialize Bedrock client (change region if needed)
bedrock = boto3.client('bedrock', region_name='us-east-1')   # Try us-east-1 or us-west-2

# Get all Qwen models available to you
response = bedrock.list_foundation_models(byProvider='qwen')

print("🔍 Available Qwen Models in your account:\n")
for model in response.get('modelSummaries', []):
    print(f"Model ID : {model['modelId']}")
    print(f"Name     : {model.get('modelName', 'N/A')}")
    print(f"Provider : {model.get('providerName', 'N/A')}")
    print("-" * 60)

🔍 Available Qwen Models in your account:

Model ID : qwen.qwen3-coder-next
Name     : Qwen3 Coder Next
Provider : Qwen
------------------------------------------------------------
Model ID : qwen.qwen3-next-80b-a3b
Name     : Qwen3 Next 80B A3B
Provider : Qwen
------------------------------------------------------------
Model ID : qwen.qwen3-32b-v1:0
Name     : Qwen3 32B (dense)
Provider : Qwen
------------------------------------------------------------
Model ID : qwen.qwen3-vl-235b-a22b
Name     : Qwen3 VL 235B A22B
Provider : Qwen
------------------------------------------------------------
Model ID : qwen.qwen3-coder-30b-a3b-v1:0
Name     : Qwen3-Coder-30B-A3B-Instruct
Provider : Qwen
------------------------------------------------------------


In [1]:
import os
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model_id="qwen.qwen3-next-80b-a3b"
)
# Test the model
response = llm.invoke("Tell me about langgraph in detail")
print(response.content)

NameError: name 'SS' is not defined

In [31]:
from anthropic import AnthropicBedrock

client = AnthropicBedrock()

response = client.messages.create(
    model="qwen.qwen3-next-80b-a3b",
    max_tokens=4096,
    temperature=0.7,
    messages=[{"role": "user", "content": "Hi How are you?"}]
)

print(response)

Message(id='chatcmpl-1f3c694c-22f4-47d6-b36c-ef9ff3d5d8a6', container=None, content=None, model='qwen.qwen3-next-80b-a3b', role=None, stop_details=None, stop_reason=None, stop_sequence=None, type=None, usage=Usage(cache_creation=None, cache_creation_input_tokens=None, cache_read_input_tokens=None, inference_geo=None, input_tokens=None, output_tokens=None, server_tool_use=None, service_tier=None, completion_tokens=29, prompt_tokens=13, total_tokens=42), choices=[{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': "Hi! 😊 I'm doing great—thanks for asking! How about you? Hope you're having a wonderful day! 🌟", 'refusal': None, 'role': 'assistant'}}], created=1779197916, object='chat.completion', service_tier='default')


In [4]:
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv(override=True)
model_name = "claude-haiku-4-5"
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]
claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=20000)
answer = response.content[0].text
print(answer)

# If a moral framework produces outcomes that reduce suffering but requires violating the core principles that justify that framework, should it be considered successful, and what would change if the framework's justification came from reducing suffering rather than from the principles themselves?


In [ ]:
import time
from datetime import datetime, timedelta
import json
import boto3
from dotenv import load_dotenv
import pytz
from agents import Agent, Runner, trace, function_tool

load_dotenv(override=True)

# ✅ Renamed to cloudwatch_client to avoid collision
cloudwatch_client = boto3.client("cloudwatch")

@function_tool
def get_cpu_metrics(instanceId: str, Instance: str = "AWS/EC2", Metric: str = "CPUUtilization", last_hours: int = 6, period: int = 1800):
    """
    Fetches CPUUtilization metrics for the User provided instance if not provided use AWS/EC2
    over the last N hours with configurable period intervals.
    """
    end_time = datetime.now(pytz.utc)
    start_time = end_time - timedelta(hours=last_hours)  

    response = cloudwatch_client.get_metric_statistics(  
        Namespace=Instance,
        MetricName=Metric,
        Dimensions=[{"Name": "InstanceId", "Value": instanceId}],
        StartTime=start_time,
        EndTime=end_time,
        Period=period,  
        Statistics=["Average"],
        Unit="Percent",
    )

    ist = pytz.timezone("Asia/Kolkata")

    for point in response["Datapoints"]:
        point["Timestamp"] = (
            point["Timestamp"].astimezone(ist).strftime("%Y-%m-%d %H:%M:%S IST")
        )

    response["Datapoints"].sort(key=lambda x: x["Timestamp"], reverse=True)
    return response


tools = [get_cpu_metrics]

prompt = "You are a cloud watcher agent that monitors cloud infrastructure."
agent = Agent(name="Chandra", instructions=prompt, model="gpt-5.4-nano", tools=tools)
result = await Runner.run(agent, "Give observations on CPU utilization for ec2 instance over last 20 hours with 30 mins timeline in detail for instanceId i-0327bf109dc40d412")

print(result.final_output)

CPUUtilization (EC2) for instance **i-0327bf109dc40d412** over **last 20 hours**, sampled every **30 minutes** (20 points expected; data returned shows **last 6 datapoints**):

- **2026-05-18 19:07 IST** — **0.267%**
- **2026-05-18 18:37 IST** — **0.277%**
- **2026-05-18 18:07 IST** — **0.295%**
- **2026-05-18 17:37 IST** — **0.266%**
- **2026-05-18 17:07 IST** — **0.267%**
- **2026-05-18 16:37 IST** — **0.265%**

### Observations
- **Sustained near-idle CPU**: values are consistently around **0.26–0.30%**.
- **No spikes / no sustained load increase**: the slight rise to **~0.295%** is minor and quickly returns to ~**0.266–0.277%**.
- **Overall trend** (based on available points): **stable and very low utilization**, indicating the instance is likely underutilized during this window.

If you want, I can re-run to ensure all **20/20** half-hour datapoints are retrieved for the full 20-hour period (the current response returned only the most recent 6).


In [17]:
import json
from datetime import datetime, timedelta
import boto3
from dotenv import load_dotenv
import pytz
from agents import Agent, Runner, trace, function_tool

load_dotenv(override=True)

# ── AWS client ────────────────────────────────────────────────────────────────
xray_client = boto3.client("xray")

# ── Tool 1: X-Ray Trace Summaries ────────────────────────────────────────────
@function_tool
def get_xray_traces(last_hours: int = 8) -> dict:
    """
    Fetches AWS X-Ray trace summaries for the last `last_hours` hours.
    Annotates each trace with Status: Fault (5xx), Error (4xx), or Success (200).
    Returns full response with TraceSummaries list.
    """
    end_time   = datetime.now(pytz.utc)
    start_time = end_time - timedelta(hours=last_hours)

    response = xray_client.get_trace_summaries(
        StartTime=start_time,
        EndTime=end_time,
    )

    ist = pytz.timezone("Asia/Kolkata")
    for t in response.get("TraceSummaries", []):
        if t.get("HasFault"):
            t["Status"] = "Fault (5xx)"
        elif t.get("HasError"):
            t["Status"] = "Error (4xx)"
        else:
            t["Status"] = "Success (200)"

        t["DurationSec"] = round(t.get("Duration", 0), 3)

        raw_ts = t.get("StartTime")
        if raw_ts and hasattr(raw_ts, "astimezone"):
            t["StartTimeIST"] = (
                raw_ts.astimezone(ist).strftime("%Y-%m-%d %H:%M:%S IST")
            )

    return response


# ── Tool 2: X-Ray Trace Details ───────────────────────────────────────────────
@function_tool
def get_xray_trace_details(trace_ids: list[str]) -> dict:
    """
    Fetches full segment details for a list of X-Ray trace IDs.
    Use this to drill down into a specific trace after get_xray_traces.
    Returns segments with subsegments, errors, and annotations.
    """
    response = xray_client.batch_get_traces(TraceIds=trace_ids)

    parsed_traces = []
    for t in response.get("Traces", []):
        segments = []
        for seg in t.get("Segments", []):
            try:
                doc = json.loads(seg["Document"])
                segments.append({
                    "name":        doc.get("name"),
                    "origin":      doc.get("origin"),
                    "start_time":  doc.get("start_time"),
                    "end_time":    doc.get("end_time"),
                    "fault":       doc.get("fault", False),
                    "error":       doc.get("error", False),
                    "http":        doc.get("http", {}),
                    "annotations": doc.get("annotations", {}),
                    "subsegments": [
                        {
                            "name":      s.get("name"),
                            "namespace": s.get("namespace"),
                            "duration":  round(
                                s.get("end_time", 0) - s.get("start_time", 0), 4
                            ),
                            "fault":     s.get("fault", False),
                            "error":     s.get("error", False),
                        }
                        for s in doc.get("subsegments", [])
                    ],
                })
            except (json.JSONDecodeError, KeyError):
                continue

        parsed_traces.append({
            "TraceId":  t.get("Id"),
            "Duration": t.get("Duration"),
            "Segments": segments,
        })

    return {"Traces": parsed_traces}


# ── Tool 3: X-Ray Service Map ─────────────────────────────────────────────────
@function_tool
def get_xray_service_map(last_hours: int = 8) -> dict:
    """
    Fetches the X-Ray service map showing all services and their connections,
    response times, fault rates, and error rates for the last `last_hours` hours.
    """
    end_time   = datetime.now(pytz.utc)
    start_time = end_time - timedelta(hours=last_hours)

    response = xray_client.get_service_graph(
        StartTime=start_time,
        EndTime=end_time,
    )

    services = []
    for svc in response.get("Services", []):
        summary_stats = svc.get("SummaryStatistics", {})
        services.append({
            "Name":            svc.get("Name"),
            "Type":            svc.get("Type"),
            "AccountId":       svc.get("AccountId"),
            "TotalRequests":   summary_stats.get("TotalCount", 0),
            "FaultCount":      summary_stats.get("FaultStatistics", {}).get("TotalCount", 0),
            "ErrorCount":      summary_stats.get("ErrorStatistics", {}).get("TotalCount", 0),
            "OkCount":         summary_stats.get("OkCount", 0),
            "AvgResponseTime": round(summary_stats.get("TotalResponseTime", 0) /
                               max(summary_stats.get("TotalCount", 1), 1), 4),
            "Edges": [
                {
                    "TargetName": e.get("TargetServiceId"),
                    "AvgLatency": round(
                        e.get("SummaryStatistics", {}).get("TotalResponseTime", 0) /
                        max(e.get("SummaryStatistics", {}).get("TotalCount", 1), 1), 4
                    ),
                }
                for e in svc.get("Edges", [])
            ],
        })

    return {"Services": services}


# ── Agent setup ───────────────────────────────────────────────────────────────
tools = [get_xray_traces, get_xray_trace_details, get_xray_service_map]

prompt = """
You are xray-tracer, an expert AWS X-Ray distributed tracing monitoring agent.
You have access to three tools:

  • get_xray_traces(last_hours)        – Fetch trace summaries; shows faults, errors, durations
  • get_xray_trace_details(trace_ids)  – Drill into specific traces by ID for segment-level detail
  • get_xray_service_map(last_hours)   – Get full service map with latency and error rates

Your monitoring workflow:
1. Always start with get_xray_traces to get the overview.
2. If faults (5xx) or errors (4xx) exist, call get_xray_trace_details on those trace IDs.
3. Call get_xray_service_map to understand service dependencies and hotspots.
4. Report findings with:
   - Total traces, fault count, error count, success rate
   - Slowest traces with their durations
   - Faulty services and which subsegments failed
   - Service map hotspots (high latency edges, high fault-rate services)
   - Actionable recommendations
5. All timestamps must be shown in IST.
"""

agent = Agent(
    name="xray-tracer",
    instructions=prompt,
    model="gpt-5.4-nano",
    tools=tools,
)

# ── Run ───────────────────────────────────────────────────────────────────────
with trace("XRay Monitoring Pipeline"):
    result = await Runner.run(
        agent,
        "Give a detailed X-Ray monitoring report for the last 8 hours. "
        "Include fault analysis, slow traces, service map insights, "
        "and actionable recommendations.",
    )

print(result.final_output)

### X-Ray Monitoring Report for Last 6 Hours

#### Overview
- **Total Traces Processed**: 0
- **Faulty Traces (5xx)**: None
- **Error Traces (4xx)**: None
- **Successful Traces (200)**: None

#### Slowest Traces
- **No trace data available** for analysis.

#### Fault Analysis
- **No faults or errors recorded** during the analyzed period.

#### Service Map Insights
- **No services or connections** detected. 

#### Actionable Recommendations
- **Review Service Configurations**: Ensure that X-Ray is properly configured within your application to capture inbound requests.
- **Verify Timeframe Settings**: Double-check if there are issues with the time synchronization or recording delays.
- **Check AWS X-Ray Setup**: The absence of trace data might indicate configuration issues with the X-Ray daemon or permissions that might prevent data collection.
- **Manual Testing**: Perform a few manual requests to see if they are getting captured and traced as expected.

Since there are no traces or se